In [ ]:
from datetime import datetime
from getpass import getpass

rdm_url = 'https://rdm.example.com/'
admin_rdm_url = 'https://admin.rdm.example.com/'
idp_name_1 = 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
idp_name_2 = 'GakuNin RDM IdP'
idp_username_2 = None
idp_password_2 = None

weko_url = 'https://weko.example.com/'
weko_admin_email = None
weko_admin_password = None
weko_user_email = None
weko_user_password = None
weko_institution_name = 'GakuNin RDM IdP'
weko_index_name = 'Sample Index'
ignore_https_errors = False

project_name_prefix = 'TEST-WEKO-{}'
oauth_application_name_prefix = 'TEST-WEKO-APP-{}'
default_result_path = None
close_on_fail = False
transition_timeout = 60000
skip_failed_test = True
exclude_notebooks = []

# WEKO SWORD API settings
weko_docker_compose_path = None
sword_mapping_id = 30002

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
if idp_username_2 is None:
    idp_username_2 = input(prompt=f'Username for {idp_name_2}')
if idp_password_2 is None:
    idp_password_2 = getpass(prompt=f'Password for {idp_username_2}@{idp_name_2}')

if weko_admin_email is None:
    weko_admin_email = input('WEKO admin email: ')
if weko_admin_password is None:
    weko_admin_password = getpass('WEKO admin password: ')
if weko_user_email is None:
    weko_user_email = input('WEKO user email: ')
if weko_user_password is None:
    weko_user_password = getpass('WEKO user password: ')
if weko_institution_name is None:
    weko_institution_name = input('機関名 (例: GakuNin RDM IdP): ')
if weko_index_name is None:
    weko_index_name = input('WEKO index name: ')

timestamp = datetime.now().strftime('%Y%m%d-%H%M')
project_name_dashboard = project_name_prefix.format(timestamp + '-1')
oauth_application_name_dashboard = oauth_application_name_prefix.format(timestamp + '-PD')
project_name_files_tab = project_name_prefix.format(timestamp + '-2')
oauth_application_name_files_tab = oauth_application_name_prefix.format(timestamp + '-FT')

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir


# GakuNinRDM 総合テスト [WEKOアドオン]

- サブシステム名: アドオン
- ページ/アドオン: WEKO
- 機能分類: プロジェクト設定
- シナリオ名: WEKOアドオン一連試験
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM, WEKO管理者、ユーザーadmin: GRDM統合管理者)


In [ ]:
from datetime import datetime
import os
from scripts.papermillHelpers import gen_run_notebook


def make_result_dir(base_path):
    result_dir = os.path.join(base_path, 'notebooks')
    os.makedirs(result_dir, exist_ok=True)
    return result_dir


result_dir = make_result_dir(default_result_path)

run_notebook = gen_run_notebook(
    result_dir,
    transition_timeout,
    dict(
        rdm_url=rdm_url,
        admin_rdm_url=admin_rdm_url,
        idp_name_1=idp_name_1,
        idp_username_1=idp_username_1,
        idp_password_1=idp_password_1,
        idp_name_2=idp_name_2,
        idp_username_2=idp_username_2,
        idp_password_2=idp_password_2,
        weko_url=weko_url,
        weko_admin_email=weko_admin_email,
        weko_admin_password=weko_admin_password,
        weko_user_email=weko_user_email,
        weko_user_password=weko_user_password,
        weko_institution_name=weko_institution_name,
        weko_index_name=weko_index_name,
        ignore_https_errors=ignore_https_errors,
    ),
    skip_failed_test,
    exclude_notebooks,
)

result_notebooks = []
result_dir

## プロジェクトダッシュボードでのテスト用の「WEKOアドオンの登録」テストの実施

プロジェクトダッシュボードでのテスト用に、テスト「テスト手順-WEKOアドオン-アドオン追加」を実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-アドオン追加.ipynb',
        dict(
            project_name=project_name_dashboard,
            oauth_application_name=oauth_application_name_dashboard,
        ),
        '-プロジェクトダッシュボード',
    )
)
result_notebooks[-1]

## プロジェクトダッシュボードでのテスト用のSWORD Client登録

プロジェクトダッシュボードでのテスト用に、WEKOにSWORD Clientを登録する。OAuthクライアント作成時に生成されたclient_idを使用する。

In [ ]:
import json
import subprocess

addon_result_dir = os.path.join(result_dir, 'テスト手順-WEKOアドオン-アドオン追加-プロジェクトダッシュボード')
oauth_info_path = os.path.join(addon_result_dir, 'weko_oauth_client.json')

with open(oauth_info_path) as f:
    oauth_info = json.load(f)
client_id_dashboard = oauth_info['client_id']
print(f'Loaded client_id: {client_id_dashboard}')

code = f'''
from weko_swordserver.api import SwordClient
from weko_swordserver.models import SwordClientModel as M
obj = SwordClient.register(
    client_id="{client_id_dashboard}",
    registration_type_id=M.RegistrationType.DIRECT,
    mapping_id={sword_mapping_id},
    active=True,
    duplicate_check=False,
)
print(f"Registered SWORD client: {{obj.client_id}}")
'''
subprocess.run(
    ['docker', 'compose', '-f', weko_docker_compose_path, 'exec', '-T', 'web',
     'invenio', 'shell', '-c', code],
    check=True
)

## プロジェクトダッシュボードでの「メタデータのWEKOへの送信-根拠データ」テストの実施

テスト「テスト手順-WEKOアドオン-メタデータ送信-根拠データ」をプロジェクトダッシュボードで実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-メタデータ送信-根拠データ.ipynb',
        dict(
            project_name=project_name_dashboard,
            use_files_tab=False,
        ),
        '-プロジェクトダッシュボード',
    )
)
result_notebooks[-1]

## プロジェクトダッシュボードでの「メタデータのWEKOへの送信-書誌情報」テストの実施

テスト「テスト手順-WEKOアドオン-メタデータ送信-書誌情報」をプロジェクトダッシュボードで実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-メタデータ送信-書誌情報.ipynb',
        dict(
            project_name=project_name_dashboard,
            use_files_tab=False,
        ),
        '-プロジェクトダッシュボード',
    )
)
result_notebooks[-1]

## プロジェクトダッシュボードでの「メタデータのWEKOへの送信-書誌情報＋根拠データ」テストの実施

テスト「テスト手順-WEKOアドオン-メタデータ送信-書誌情報＋根拠データ」をプロジェクトダッシュボードで実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-メタデータ送信-書誌情報＋根拠データ.ipynb',
        dict(
            project_name=project_name_dashboard,
        ),
        '-プロジェクトダッシュボード',
    )
)
result_notebooks[-1]

## ファイルタブでのテスト用の「WEKOアドオンの登録」テストの実施

ファイルタブでのテスト用に、テスト「テスト手順-WEKOアドオン-アドオン追加」を実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-アドオン追加.ipynb',
        dict(
            project_name=project_name_files_tab,
            oauth_application_name=oauth_application_name_files_tab,
        ),
        '-ファイルタブ',
    )
)
result_notebooks[-1]

## ファイルタブでのテスト用のSWORD Client登録

ファイルタブでのテスト用に、WEKOにSWORD Clientを登録する。OAuthクライアント作成時に生成されたclient_idを使用する。

In [ ]:
addon_result_dir = os.path.join(result_dir, 'テスト手順-WEKOアドオン-アドオン追加-ファイルタブ')
oauth_info_path = os.path.join(addon_result_dir, 'weko_oauth_client.json')

with open(oauth_info_path) as f:
    oauth_info = json.load(f)
client_id_files_tab = oauth_info['client_id']
print(f'Loaded client_id: {client_id_files_tab}')

code = f'''
from weko_swordserver.api import SwordClient
from weko_swordserver.models import SwordClientModel as M
obj = SwordClient.register(
    client_id="{client_id_files_tab}",
    registration_type_id=M.RegistrationType.DIRECT,
    mapping_id={sword_mapping_id},
    active=True,
    duplicate_check=False,
)
print(f"Registered SWORD client: {{obj.client_id}}")
'''
subprocess.run(
    ['docker', 'compose', '-f', weko_docker_compose_path, 'exec', '-T', 'web',
     'invenio', 'shell', '-c', code],
    check=True
)

## ファイルタブでの「メタデータのWEKOへの送信-根拠データ」テストの実施

テスト「テスト手順-WEKOアドオン-メタデータ送信-根拠データ」をファイルタブで実施する。

In [ ]:
result_notebooks.append(
    run_notebook(
        'テスト手順-WEKOアドオン-メタデータ送信-根拠データ.ipynb',
        dict(
            project_name=project_name_files_tab,
            use_files_tab=True,
        ),
        '-ファイルタブ',
    )
)
result_notebooks[-1]

終了処理を実施。


In [ ]:
!rm -fr {work_dir}
